In [ ]:
from snowflake.snowpark.context import get_active_session

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
session = get_active_session()

df = session.table(
    "DISCRETE_MFG_COST_MODEL.CORE_ML.TDS_TRAINING_DATA"
).to_pandas()

df.head()

In [ ]:
len(df)

In [ ]:
X = df.drop(
    columns=[
        "TOP_LEVEL_ITEM",
        "REVISION",
        "TARGET_TDS"
    ]
)

y = df["TARGET_TDS"]

In [ ]:
df["TARGET_TDS"].isna().sum()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
model = RandomForestRegressor(

    n_estimators=300,

    max_depth=10,

    random_state=42

)

model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, pred))

print("RMSE :", mean_squared_error(y_test, pred) ** 0.5)

print("R2 :", r2_score(y_test, pred))

In [ ]:
import joblib

joblib.dump(model, "tds_random_forest.pkl")

In [ ]:
import snowflake.ml

from importlib.metadata import version
print(version('snowflake-ml-python'))

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="DISCRETE_MFG_COST_MODEL",
    schema_name="CORE_ML"
)

In [ ]:
sample_input = X_train.head(5)

In [ ]:
print(sample_input.columns)
print(sample_input.shape)

In [ ]:
model_version = registry.log_model(
    model=model,
    model_name="TDS_MODEL",
    version_name="V2",
    comment="Random Forest Tooling Degradation Score",
    sample_input_data=sample_input
)

In [ ]:
ai_df = session.table(
    "DISCRETE_MFG_COST_MODEL.CORE_FEATURES.AI_CONTEXT_V"
).to_pandas()

In [ ]:
feature_cols = X_train.columns.tolist()

X_predict = ai_df[feature_cols]

In [ ]:
ai_df["PREDICTED_TDS"] = model.predict(X_predict)

In [ ]:
ai_df[
    [
        "TOP_LEVEL_ITEM",
        "PREDICTED_TDS"
    ]
]

In [ ]:
tds_results = ai_df[
    [
        "TOP_LEVEL_ITEM",
        "REVISION"
    ]
].copy()

tds_results["RAW_TDS"] = model.predict(X_predict).round(3)

In [ ]:
# Convert raw TDS into 0-100 risk score
tds_results["TDS"] = (
    (
        (tds_results["RAW_TDS"] - 1.5)
        /
        (3.5 - 1.5)
    ) * 100
).clip(0, 100).round(2)

In [ ]:
def tooling_risk(score):
    if score < 25:
        return "LOW"
    elif score < 50:
        return "MODERATE"
    elif score < 75:
        return "HIGH"
    else:
        return "CRITICAL"

tds_results["TOOLING_RISK"] = tds_results["TDS"].apply(tooling_risk)

In [ ]:
import numpy as np

tree_predictions = np.array(
    [tree.predict(X_predict) for tree in model.estimators_]
)

confidence = 100 - (
    tree_predictions.std(axis=0) /
    np.maximum(tree_predictions.mean(axis=0), 0.01)
) * 100

confidence = np.clip(confidence, 50, 99)

tds_results["MODEL_CONFIDENCE"] = confidence.round(2)

In [ ]:
session.write_pandas(
    tds_results,
    table_name="TDS_RESULTS",
    database="DISCRETE_MFG_COST_MODEL",
    schema="CORE_OUTPUT",
    overwrite=True,
    auto_create_table=True
)